# GMM 目标函数的分布

> 对应代码：`MathStatsCode/code_in_notes/Chap.10/GMM_mini_chi2.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：模拟 Poisson(10) 数据，比较 GMM 目标函数在估计值处（渐近 χ²(1)）与真值处的分布。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.10`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.10")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

本 notebook 模拟 **GMM 目标函数（J 统计量）在估计值处与真值处的分布**，直观展示“过度识别检验”的原理。

**①定义估计程序 `est_lambda_chi2`**：`syntax varname [, lambda(1) lambda0(10) n(100)]`；用 `gmm (x-{lambda}) (x²-({lambda}+{lambda}²)), winit(identity)` 对 Poisson(λ) 的两个理论矩（一阶矩 λ、二阶矩 λ+λ²）做 GMM——一个参数、两个矩，属于**过度识别**。`e(J)` 取出估计值处的目标函数值，`e(W)` 取出估计时使用的权重矩阵。

**②在真值处重算目标函数**：构造 `m1_0=x-λ0`、`m2_0=x²-(λ0+λ0²)`（真值处的矩条件），求和得到矩条件的样本均值向量 g，再计算二次型 `objf = g'Wg`、`chi2_true = objf/n`——即用与估计时相同的权重矩阵、但固定在真值处评价目标函数。

**③模拟2000次**：每次生成100个 Poisson(10) 观测（`lambda0(10)`），记录估计值处与真值处的目标函数值；画出两个直方图分别与 χ²(1)、χ²(2) 理论密度对比。

**统计原理**：恰好识别时目标函数恒为0；**过度识别**时，估计值处的目标函数服从自由度=矩条件数-参数个数=2-1=1 的 χ² 分布（这是 J 检验的基础）；而真值处的目标函数服从自由度=矩条件数=2 的 χ²——两个分布相差一个被“估计掉”的自由度，图中直观可见。


In [ ]:
%%stata
clear all
set more off
set seed 20231213
cap program drop est_lambda_chi2
program est_lambda_chi2, rclass
	syntax varname [, lambda(real 1) lambda0(real 10) n(integer 100)]
	tempvar x2 m1_0 m2_0
	tempname mean_m1_0 mean_m2_0 vec_g chi2_est chi2_true wmatrix objf
	quietly{
		gen `x2'=`varlist'^2
		gmm (`varlist'-{lambda=`lambda'}) (`x2'-({lambda}+{lambda}^2)), winit(identity)
		local `chi2_est'=e(J)
		mat `wmatrix'=e(W)
		gen `m1_0'=`varlist'-`lambda0'
		gen `m2_0'=`x2'-(`lambda0'+`lambda0'^2)
		su `m1_0'
		local `mean_m1_0'=r(sum)
		su `m2_0'
		local `mean_m2_0'=r(sum)
		mat `vec_g'=(``mean_m1_0'' \ ``mean_m2_0'')
		mat `objf'=`vec_g''*`wmatrix'*`vec_g'
		local `chi2_true'=`objf'[1,1]/`n'
	}
	return scalar chi2_est = ``chi2_est''
	return scalar chi2_true = ``chi2_true''
end

frame create simulation chi2_est chi2_true
forvalues i=1/2000{
	quietly{
		clear
		set obs 100
		gen x=rpoisson(10)
		est_lambda_chi2 x, lambda0(10) n(100)
		frame post simulation (r(chi2_est)) (r(chi2_true))
	}
}
frame change simulation
label variable chi2_est "估计值处的目标函数"
label variable chi2_true "真值处的目标函数"
gen density_chi2_1=chi2den(1,chi2_est)
label variable density_chi2_1 "Chi(1)密度函数"
gen density_chi2_2=chi2den(2,chi2_est)
label variable density_chi2_2 "Chi(2)密度函数"
sort chi2_est
twoway  (hist chi2_est, density) (line density_chi2_1 chi2_est  if chi2_est>0.3) (line density_chi2_2 chi2_est  if chi2_est>0.3), legend(pos(1) ring(0) label(1 "估计值处的目标函数密度")) xtitle("")
twoway  (hist chi2_true, density) (line density_chi2_1 chi2_est  if chi2_est>0.3) (line density_chi2_2 chi2_est  if chi2_est>0.3), legend(pos(1) ring(0) label(1 "真值处的目标函数密度")) xtitle("")
